In [1]:
import os
import sys
sys.path.append('..')
import torch
import librosa
from scipy.io import wavfile
from IPython import display as disp
from pathlib import Path
from tqdm import tqdm

from src.modules import MimiResynthesisModule, MimiKMResynthesisModule, MimiResidualKMResynthesisModule, SpeakerModule
from anonymization.anonymizer import DemoAnonymizer

/home/grads/q/quamer.waris/anaconda3/envs/xarts/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [allow_tf32, disable_jit_profiling]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
/mnt/nvme-data1/waris/PSI-TAMU/DarkStream/notebooks/../anonymization/demo_speaker_embeddings.py:5: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier


In [2]:

os.environ["CUDA_VISIBLE_DEVICES"] = "3" # Set the GPU device
MAX_WAV_VALUE = 32768.0

### Note
Use mimiresynthesis_nola for 0ms lookahead, mimiresynthesis_la7 for 140ms lookahead, and mimiresynthesis_la14 for 280ms lookahead model

MimiResynthesisModule -> for wav + CL archiecture

MimiKMResynthesisModule -> for wav + CL + KM architecture

In [ ]:
speaker_encoder = SpeakerModule(device="cuda")
# resynthesizer = MimiResynthesisModule.load_from_checkpoint(
#     "../ckpts/mimiresynthesis_la14.ckpt",
#     encoder_path=None,
#     map_location="cuda"
# )

# Uncomment the following line to use the KM Resynthesis Module instead
# resynthesizer = MimiKMResynthesisModule.load_from_checkpoint(
#     "../ckpts/mimikmresynthesis_la7.ckpt",
#     encoder_path=None,
#     kmeans_path=None,
#     map_location="cuda"
# )


# Uncomment the following line to use the Residual KM Resynthesis Module instead
resynthesizer = MimiResidualKMResynthesisModule.load_from_checkpoint(
    "../ckpts/mimiresidualkmresynthesis_la7_nq4.ckpt",  # Supports nq2, nq4, nq8. ajust paths accordingly.
    encoder_path=None,
    kmeans_path="../ckpts/km256_centers.pth",
    map_location="cuda"
)

resynthesizer.eval()

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
/home/grads/q/quamer.waris/anaconda3/envs/xarts/lib/python3.10/site-packages/speechbrain/utils/autocast.py:68: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cac

MimiResidualKMResynthesisModule(
  (encoder): MimiPasrModule(
    (net): MimiEncoderModel(
      (encoder): SEANetEncoder(
        (model): Sequential(
          (0): StreamingConv1d(
            (conv): NormConv1d(
              (conv): RawStreamingConv1d(1, 64, kernel_size=(7,), stride=(1,))
            )
          )
          (1): SEANetResnetBlock(
            (block): Sequential(
              (0): ELU(alpha=1.0)
              (1): StreamingConv1d(
                (conv): NormConv1d(
                  (conv): RawStreamingConv1d(64, 32, kernel_size=(3,), stride=(1,))
                )
              )
              (2): ELU(alpha=1.0)
              (3): StreamingConv1d(
                (conv): NormConv1d(
                  (conv): RawStreamingConv1d(32, 64, kernel_size=(1,), stride=(1,))
                )
              )
            )
            (add): StreamingAdd()
            (shortcut): Identity()
          )
          (2): ELU(alpha=1.0)
          (3): StreamingConv1d(
       

In [4]:
wav_file = "../sample_wav/BDL/arctic_a0001.wav"
_device = "cuda"
sample_rate = 16000

audio, _ = librosa.load(wav_file, sr=sample_rate)

disp.Audio(audio, rate=16000)

### Non streaming synthesis

In [5]:
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(_device)
embed = speaker_encoder(audio.squeeze(0))
resyth_audio = resynthesizer(audio, embed)
resyth_audio = resyth_audio.squeeze()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.detach().cpu().numpy().astype('int16')

disp.Audio(resyth_audio, rate=16000)

### VC


In [6]:
tgt_wav_file = "../sample_wav/SLT/arctic_a0001.wav"
tgtaudio, _ = librosa.load(tgt_wav_file, sr=sample_rate)
disp.Audio(tgtaudio, rate=16000)

In [7]:
tgtaudio = torch.from_numpy(tgtaudio).unsqueeze(0).unsqueeze(0).to(_device)

embed = speaker_encoder(tgtaudio.squeeze(0))
vc_audio = resynthesizer(audio, embed)
vc_audio = vc_audio.squeeze()
vc_audio = vc_audio * MAX_WAV_VALUE
vc_audio = vc_audio.detach().cpu().numpy().astype('int16')

disp.Audio(vc_audio, rate=16000)

### Streaming synthesis (One way)

In [19]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    frame_size = 320 # 20ms for 16000Hz sample rate, incraease to increase chunk size

    # pad the end of audio so lookahead doesn't drop the last few frames
    lookahead_size = 14
    pad = torch.zeros((1, 1, lookahead_size * frame_size)).to(audio.device)
    audio_padded = torch.cat([audio, pad], dim=-1).to(audio.device)

    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):  # in steps of frame_size, so chunk size is 20ms
                frame = audio_padded[:, :, offset:offset + frame_size]
                z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

  0%|          | 0/162 [00:00<?, ?it/s]

100%|██████████| 162/162 [00:02<00:00, 56.09it/s]


### Streaming synthesis (Another way)

In [ ]:
resynthesizer.encoder.net.streaming_forever(batch_size=1)
resynthesizer.decoder.streaming_forever(batch_size=1)
with torch.no_grad():
    original_length = audio.shape[-1]
    frame_size = 320

    all_audios = []
    all_z_embeds = []

    for offset in tqdm(range(0, original_length, frame_size)):
        frame = audio[:, :, offset:offset + frame_size]
        z_embed = resynthesizer.encoder.net(frame)
        if z_embed.shape[-1] == 0:
            continue
        resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, embed)
        all_audios.append(resynth_audio_chunk)   
        all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 162/162 [00:02<00:00, 56.58it/s]


### Anonymization
For anonymization use embedding from anonymizer model instead

In [16]:
anonymizer = DemoAnonymizer(
    model_path=Path('../anonymization/models'), 
    model_tag='gan',
    device="cuda"
)

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Model type of anonymizer: gan


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding

In [17]:
anon_embed = anonymizer.anonymize_embedding(audio.squeeze().cpu().numpy(), 16000)
anon_embed = torch.cat([anon_embed[192:], anon_embed[:192]], dim=0) #embeds are concatenated in the reversed order

In [18]:
resynthesizer.eval()
with torch.no_grad():
    original_length = audio.shape[-1]
    # Pad audio for lookahead
    # frame_size = int(resynthesizer_mm.hparams.sampling_rate // resynthesizer_mm.hparams.frame_rate)
    frame_size = 320

    # frame_size = int(encoder_mm.net.sample_rate // encoder_mm.net.frame_rate)
    all_audios = []
    all_z_embeds = []

    with resynthesizer.encoder.net.streaming(batch_size=1):
        with resynthesizer.decoder.streaming(batch_size=1):
            for offset in tqdm(range(0, original_length, frame_size)):
                frame = audio[:, :, offset:offset + frame_size]
                z_embed = resynthesizer.encoder.net(frame)
                if z_embed.shape[-1] == 0:
                    continue
                resynth_audio_chunk = resynthesizer.decoder.decode(z_embed, anon_embed.unsqueeze(0))
                all_audios.append(resynth_audio_chunk)   
                all_z_embeds.append(z_embed)    
    
all_audios = torch.cat(all_audios, dim=-1)
resyth_audio = all_audios.squeeze().cpu().numpy()
resyth_audio = resyth_audio * MAX_WAV_VALUE
resyth_audio = resyth_audio.astype('int16')

disp.Audio(resyth_audio, rate=16000)

  0%|          | 0/162 [00:00<?, ?it/s]

100%|██████████| 162/162 [00:03<00:00, 50.44it/s]
